Antes Preparacion del modelo

In [1]:
# PSEUDOCODIGO:
# Preparacion, entrenamiento y validacion del modelo

from yolov8 import YOLOv8
from ocr_library import PaddleOCR
import numpy as np

# Paso 1: Cargar dataset de imagenes anotadas
def preparar_dataset():
    imagenes = cargar_imagenes_desde_banco_datos('dataset_matriculas/')
    anotaciones = cargar_anotaciones('labels.json')
    # Dividir en train, val, test (80%, 10%, 10%)
    train_set, val_set, test_set = dividir_dataset(imagenes, anotaciones)
    return train_set, val_set, test_set

# Paso 2: Preprocesamiento de imagenes
def preprocesar_imagen(imagen):
    imagen_redimensionada = redimensionar(imagen, (640, 640))
    imagen_normalizada = normalizar(imagen_redimensionada)
    return imagen_normalizada

# Paso 3: Entrenamiento del modelo
def entrenar_modelo(train_set):
    modelo = YOLOv8('yolov8n.pt')  # Modelo preentrenado
    for epoca in range(50):
        for lote in train_set:
            imagen_preprocesada = preprocesar_imagen(lote['imagen'])
            prediccion = modelo.predecir(imagen_preprocesada)
            perdida = calcular_perdida(prediccion, lote['etiqueta'])
            actualizar_pesos(modelo, perdida)
    return modelo

# Paso 4: Validacion del modelo
def validar_modelo(modelo, val_set):
    confianzas = []
    for lote in val_set:
        imagen_preprocesada = preprocesar_imagen(lote['imagen'])
        prediccion = modelo.predecir(imagen_preprocesada)
        confianza = obtener_confianza(prediccion)
        confianzas.append(confianza)
    precision_promedio = calcular_metrica(confianzas)
    return precision_promedio

# Paso 5: Guardar modelo entrenado
def guardar_modelo(modelo):
    modelo.guardar('modelo_matriculator_v1.pt')
    print('Modelo guardado exitosamente')

# Ejecutar pipeline de preparacion
train_set, val_set, test_set = preparar_dataset()
modelo_entrenado = entrenar_modelo(train_set)
precision = validar_modelo(modelo_entrenado, val_set)
guardar_modelo(modelo_entrenado)
print(f'Precision en validacion: {precision:.2%}')

ModuleNotFoundError: No module named 'yolov8'

Despues preparacion modelo

In [ ]:
# PSEUDOCODIGO: DESPUES de integrar el modelo en la aplicacion
# Flujo de inferencia cuando el usuario interactua con la aplicacion

from yolov8 import YOLOv8
from ocr_library import PaddleOCR

# Paso 1: Recepcion de imagen desde interfaz web
def recibir_imagen(archivo_usuario):
    imagen = cargar_imagen(archivo_usuario)
    return imagen

# Paso 2: Validacion de formato
def validar_formato(imagen):
    try:
        assert imagen.formato in ['JPG', 'PNG']
        assert imagen.tamaño < 10_MB
        return True
    except AssertionError:
        return False  # Enviar mensaje error al usuario

# Paso 3: Preprocesamiento
def preprocesar_imagen(imagen):
    imagen_redimensionada = redimensionar(imagen, (640, 640))
    imagen_normalizada = normalizar(imagen_redimensionada)
    return imagen_normalizada

# Paso 4: Cargar modelo entrenado
def cargar_modelo():
    modelo = YOLOv8.cargar('modelo_matriculator_v1.pt')
    ocr = PaddleOCR(use_gpu=True)
    return modelo, ocr

# Paso 5: Inferencia (deteccion + OCR)
def inferencia(imagen_preprocesada, modelo, ocr):
    # Deteccion de matricula (bounding box)
    detecciones = modelo.predecir(imagen_preprocesada)
    caja_matricula = detecciones[0]  # Coordenadas x1, y1, x2, y2
    confianza_deteccion = detecciones.confianza

    # Extraer region de matricula
    matricula_crop = imagen_preprocesada[caja_matricula.y1:caja_matricula.y2, caja_matricula.x1:caja_matricula.x2]

    # OCR para leer caracteres
    texto_matricula = ocr.leer_texto(matricula_crop)
    confianza_ocr = ocr.confianza

    return {
        'matricula': texto_matricula,
        'confianza': min(confianza_deteccion, confianza_ocr),
        'caja': caja_matricula
    }

# Paso 6: Evaluacion de confianza
def evaluar_confianza(resultado):
    umbral = 0.85  # 85%
    return resultado['confianza'] >= umbral

# Paso 7: Bifurcacion segun confianza
def procesar_resultado(resultado, confianza_valida):
    if confianza_valida:
        # Registro automatico
        guardar_registro(resultado, automatico=True)
        mensaje = f'Matricula detectada: {resultado["matricula"]}'
    else:
        # Derivar a revision humana
        guardar_pendiente_revision(resultado)
        mensaje = 'Confianza baja. Requiere revision manual.'
    return mensaje

# Paso 8: Salida final
def mostrar_resultado(mensaje, resultado):
    enviar_respuesta_web({
        'mensaje': mensaje,
        'matricula': resultado.get('matricula'),
        'confianza': round(resultado['confianza'] * 100, 2)
    })

# FLUJO PRINCIPAL
imagen = recibir_imagen(archivo_usuario)

if not validar_formato(imagen):
    mostrar_resultado('Error: formato no valido', {})
else:
    imagen_prep = preprocesar_imagen(imagen)
    modelo, ocr = cargar_modelo()
    resultado = inferencia(imagen_prep, modelo, ocr)
    confianza_ok = evaluar_confianza(resultado)
    mensaje = procesar_resultado(resultado, confianza_ok)
    mostrar_resultado(mensaje, resultado)